# DocSmile Qwen Text Eval (Lightning AI)

Runs the existing `scripts/run_text_eval.py` pipeline on the frozen text-only eval pack.

**Recommended runtime:** Lightning AI Studio with GPU. `L4`, `A100`, or `A10G`-class GPUs are safest for `Qwen/Qwen2.5-7B`. A `T4` may require smaller batches or quantized weights.

## 1) Set project path

Upload or `git clone` the repo into your Lightning AI workspace. If you opened this notebook from inside the repo, `PROJECT_DIR` will auto-detect it; otherwise set it to the repo folder.

In [ ]:
from pathlib import Path

# If this notebook is inside the repo, this will resolve correctly.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "scripts").exists() and (PROJECT_DIR / "DocSmile").exists():
    PROJECT_DIR = PROJECT_DIR / "DocSmile"

MODEL_ID = "Qwen/Qwen2.5-7B"
OUTPUT_DIR = "evals/results/qwen2_5_7b_base_lightning"
MAX_ROWS = 0  # set >0 for a smoke test, e.g. 20


In [ ]:
import os

os.chdir(PROJECT_DIR)
!pwd
!nvidia-smi


## 2) Install runtime dependencies

In [ ]:
!pip -q install -U torch transformers accelerate sentencepiece


## 3) Verify the eval pack

This confirms the three frozen text-only datasets are present before launching the full run.

In [ ]:
import json
from pathlib import Path

eval_dir = Path(PROJECT_DIR) / 'evals' / 'text_only'
manifest = json.loads((eval_dir / 'manifest.json').read_text(encoding='utf-8'))
manifest


In [ ]:
!ls -lh evals/text_only


## 4) Run the baseline eval

This uses the repo's existing evaluator unchanged. Set `MAX_ROWS` above to a small number first if you want a quick smoke test.

In [ ]:
import os
import shlex
from pathlib import Path

script_path = Path(PROJECT_DIR) / "scripts" / "run_text_eval.py"
eval_dir = Path(PROJECT_DIR) / "evals" / "text_only"
if not eval_dir.exists() and (Path(PROJECT_DIR) / "text_only").exists():
    eval_dir = Path(PROJECT_DIR) / "text_only"

cmd = [
    "python", str(script_path),
    "--eval-dir", str(eval_dir),
    "--output-dir", OUTPUT_DIR,
    "--model", MODEL_ID,
    "--backend", "local-hf",
    "--dtype", "auto",
 ]
if MAX_ROWS > 0:
    cmd += ["--max-rows", str(MAX_ROWS)]
print(" ".join(shlex.quote(part) for part in cmd))
os.system(" ".join(shlex.quote(part) for part in cmd))


## 5) Load the summary and inspect outputs

In [ ]:
import json
from pathlib import Path

summary_path = Path(PROJECT_DIR) / OUTPUT_DIR / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary


In [ ]:
for result in summary['results']:
    print(result['dataset'])
    print('  rows:', result['rows'])
    if result['mcq_accuracy'] is not None:
        print('  mcq_accuracy:', result['mcq_accuracy'])
    print('  predictions:', result['predictions'])
    print()


## 6) Peek at a few model answers

Useful for manual QA on the open-ended datasets before you add judge-model scoring.

In [ ]:
import json
from itertools import islice
from pathlib import Path

pred_path = Path(PROJECT_DIR) / OUTPUT_DIR / 'oral_disease_open_qa_predictions.jsonl'
with pred_path.open('r', encoding='utf-8') as handle:
    rows = [json.loads(line) for line in islice(handle, 3)]
rows


## 7) Package the result folder

In [ ]:
!zip -r qwen2_5_7b_base_lightning_results.zip $OUTPUT_DIR


## Notes

- `medmcqa_dental_mcq.jsonl` gets automatic accuracy in `summary.json`.
- The two open-QA files produce predictions only; score them later with your 0-5 rubric or a judge model.
- If the session restarts, rerun from the eval command cell; outputs are written under `OUTPUT_DIR` in the workspace.